In [ ]:
# Basic imports and method to write autoscaling results
from numpy import percentile as percentile
from ascal import Ascal

def plot_ascal_results(ascal_problem: Ascal, exp_name: str):
    # Get application overloads as workload/performance
    workloads = ascal_problem.get_workloads()
    performances = ascal_problem.get_performances()
    overloads = {app: [(w-p)/w for w, p in zip(workloads[app], performances[app])] for app in workloads}

    # Get queue waiting times relative to service times, assuming each container is a server in a 
    # heterogenous D/D/n queue
    relative_queue_waiting_times = ascal_problem.get_relative_queue_waiting_times()
    avgs = {
        app_name: sum(waiting_times) / len(waiting_times)
        for app_name, waiting_times in relative_queue_waiting_times.items()
    }
    for app_name in dict(relative_queue_waiting_times):
        relative_queue_waiting_times[f"{app_name} avg = {avgs[app_name]:.6f}"] =\
            relative_queue_waiting_times.pop(app_name)
    # Plot autoscaling information
    ascal_problem.plot(ascal_problem.get_performances(), f"Application Performances ({exp_name})", "req/s")
    cluster_cost = ascal_problem.get_cluster_cost()
    total_cost_str = f"total cost = {sum(cluster_cost)/3600:.3f} $"
    ascal_problem.plot({total_cost_str: cluster_cost}, f"Cluster Cost ({exp_name})", "$/hour")
    ascal_problem.plot(overloads, f"Application Overloads ({exp_name})")
    ascal_problem.plot(relative_queue_waiting_times, f"Relative queue waiting times ({exp_name})")

In [ ]:
"""
Generate the solution for any YAML file in problem's directory.
The generated solutions, stored in solution's directory will be compared with those obtained later from tests.
"""

import os
import shutil
from pathlib import Path
from fcma import Vm
from ascal import AscalConfig, Ascal
import aws_eu_west_1_c5m5r5

PROBLEMS_DIR = "../problems"
NEW_SOLUTIONS_DIR = "../new-solutions"

# Create the new solutions directory
if os.path.exists(NEW_SOLUTIONS_DIR):
    shutil.rmtree(NEW_SOLUTIONS_DIR)  # Remove the entire directory and its contents
os.makedirs(NEW_SOLUTIONS_DIR)  # Recreate the empty directory

# List of problem YAML files
problem_files = [f.name for f in Path(PROBLEMS_DIR).glob('*.yaml') if f.is_file()]

# Reset VM IDs to ensure unique IDs for each run

# Generate problem solutions
for problem_file in problem_files:
    Vm.reset_ids()
    file_path = f"{PROBLEMS_DIR}/{problem_file}"
    print(f"\n{'-'*50}\nSolving problem {file_path}\n{'-'*50}")
    ascal_config = AscalConfig.get_from_config_yaml(file_path, aws_eu_west_1_c5m5r5.c5_m5_r5_fm)
    problem_name = f"{NEW_SOLUTIONS_DIR}/{problem_file[:-len('.yaml')] }"
    sol_file = problem_name + "-alloc.yaml"
    log_file = problem_name + ".log" # Log files are not used for testing, but to explain allocation differences
    ascal_problem = Ascal(ascal_config, log=log_file)
    ascal_problem.run()
    ascal_problem.write_allocations(f"{NEW_SOLUTIONS_DIR}/{sol_file}")

    # Print the results
    plot_ascal_results(ascal_problem, problem_file[:-len('.yaml')])
